# OncoBind-ML — Linear Regression Track
## Model A → Model B → Model C

Run the notebook from top to bottom.

- **Model A:** LIG + MUT + ENV
- **Model B:** Model A + PLIP
- **Model C:** Model B + VINA + P2

The same locked test rows are used for A, B and C.


## Common setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from scipy import stats
from scipy.stats import pearsonr, spearmanr

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend" / "app").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook inside the OncoBind-ML project folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEED = 42
TARGET = "DDG.EXP"
DATA_PATH = Path(os.environ.get("ONCOBIND_DATA_PATH", str(PROJECT_ROOT / "data" / "raw" / "MdrDB_Full_Scraped.csv")))
if not DATA_PATH.is_file() or DATA_PATH.stat().st_size == 0:
    raise FileNotFoundError(f"Upload the nonempty MdrDB dataset to {DATA_PATH}")
N_JOBS = int(os.environ.get("ONCOBIND_N_JOBS", "1"))
np.random.seed(SEED)


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.shape

In [ ]:
LIG = [c for c in df.columns if c.startswith("LIG.")]
MUT = [c for c in df.columns if c.startswith("MUT.")]
ENV = [c for c in df.columns if c.startswith("ENV.")]
PLIP = [c for c in df.columns if c.startswith("PLIP.")]
VINA = [c for c in df.columns if c == "vina" or (c.startswith("vina") and c[4:].isdigit())]
P2 = [c for c in df.columns if c.startswith(("P2.", "P2dl.", "P2dp."))]

In [ ]:
A_FEATURES = LIG + MUT + ENV
B_FEATURES = A_FEATURES + PLIP
C_FEATURES = B_FEATURES + VINA + P2

pd.Series({
    "A": len(A_FEATURES),
    "B": len(B_FEATURES),
    "C": len(C_FEATURES)
})

In [ ]:
y = pd.to_numeric(df[TARGET], errors="coerce")
valid = y.notna()

df_clean = df.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)

len(df_clean)

In [ ]:
all_idx = np.arange(len(df_clean))

dev_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=SEED
)

y_dev = y.iloc[dev_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

len(dev_idx), len(test_idx)

In [ ]:
def metrics(y_true, pred):
    return {
        "R2": r2_score(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MAE": mean_absolute_error(y_true, pred),
        "Pearson": pearsonr(y_true, pred).statistic,
        "Spearman": spearmanr(y_true, pred).statistic
    }

In [ ]:
def actual_vs_pred(y_true, pred, title):
    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(y_true, pred, s=10, alpha=0.3)
    lo = min(np.min(y_true), np.min(pred))
    hi = max(np.max(y_true), np.max(pred))
    plt.plot([lo, hi], [lo, hi], "--")
    plt.xlabel("Actual DDG.EXP")
    plt.ylabel("Predicted DDG.EXP")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def residual_plot(pred, residual, title):
    plt.figure(figsize=(6.5, 4))
    plt.scatter(pred, residual, s=10, alpha=0.3)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted DDG.EXP")
    plt.ylabel("Residual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# MODEL A — Linear Regression
### LIG + MUT + ENV

Model A is the clean OLS baseline. First understand the data, then train and diagnose Linear Regression.

In [ ]:
XA = df_clean[A_FEATURES].apply(pd.to_numeric, errors="coerce")
XA = XA.replace([np.inf, -np.inf], np.nan)
XA.shape

In [ ]:
XA_dev = XA.iloc[dev_idx].reset_index(drop=True)
XA_test = XA.iloc[test_idx].reset_index(drop=True)

In [ ]:
pd.Series({
    "Development rows": len(XA_dev),
    "Test rows": len(XA_test),
    "Features": XA_dev.shape[1]
})

In [ ]:
A_missing = XA_dev.isna().mean().sort_values(ascending=False)
A_missing.head(10).to_frame("Missing")

In [ ]:
tmp = A_missing.head(12).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(tmp.index, tmp.values)
plt.xlabel("Missing fraction")
plt.title("Model A missing values")
plt.tight_layout()
plt.show()

In [ ]:
A_unique = XA_dev.nunique(dropna=True)
A_constant = A_unique[A_unique <= 1].index.tolist()
A_constant

In [ ]:
A_KEEP = [c for c in XA_dev.columns if c not in A_constant]
len(A_KEEP)

In [ ]:
XA_dev = XA_dev[A_KEEP]
XA_test = XA_test[A_KEEP]

In [ ]:
A_imputer = SimpleImputer(strategy="median")

In [ ]:
XA_dev_imp = pd.DataFrame(
    A_imputer.fit_transform(XA_dev),
    columns=A_KEEP
)

In [ ]:
XA_test_imp = pd.DataFrame(
    A_imputer.transform(XA_test),
    columns=A_KEEP
)

In [ ]:
XA_dev_imp.describe().T[["mean", "std", "min", "max"]].head(10).round(3)

In [ ]:
A_skew = XA_dev_imp.skew().sort_values(key=np.abs, ascending=False)
A_skew.head(12).to_frame("Skewness")

In [ ]:
tmp = A_skew.head(12).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(tmp.index, tmp.values)
plt.xlabel("Skewness")
plt.title("Model A skewness")
plt.tight_layout()
plt.show()

In [ ]:
A_corr = XA_dev_imp.corr().abs()
A_upper = A_corr.where(
    np.triu(np.ones(A_corr.shape), k=1).astype(bool)
)

In [ ]:
A_pairs = A_upper.stack().sort_values(ascending=False)
A_pairs.head(15)

In [ ]:
A_scaler = StandardScaler()

In [ ]:
XA_dev_scaled = pd.DataFrame(
    A_scaler.fit_transform(XA_dev_imp),
    columns=A_KEEP
)

In [ ]:
XA_test_scaled = pd.DataFrame(
    A_scaler.transform(XA_test_imp),
    columns=A_KEEP
)

In [ ]:
XA_dev_scaled.describe().T[["mean", "std"]].head(10).round(3)

In [ ]:
A_model = LinearRegression()

In [ ]:
A_model.fit(XA_dev_scaled, y_dev)

In [ ]:
A_train_pred = A_model.predict(XA_dev_scaled)

In [ ]:
A_test_pred = A_model.predict(XA_test_scaled)

In [ ]:
A_result = metrics(y_test, A_test_pred)
pd.Series(A_result).round(4)

In [ ]:
pd.DataFrame({
    "Split": ["Development", "Locked test"],
    "R2": [
        r2_score(y_dev, A_train_pred),
        r2_score(y_test, A_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_dev, A_train_pred)),
        np.sqrt(mean_squared_error(y_test, A_test_pred))
    ]
}).round(4)

In [ ]:
A_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

In [ ]:
A_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [ ]:
A_cv_score = cross_validate(
    A_pipe,
    XA.iloc[dev_idx][A_KEEP],
    y.iloc[dev_idx],
    cv=A_cv,
    scoring={
        "R2": "r2",
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error"
    },
    n_jobs=N_JOBS
)

In [ ]:
A_cv_table = pd.DataFrame({
    "Fold": np.arange(1, 6),
    "R2": A_cv_score["test_R2"],
    "RMSE": -A_cv_score["test_RMSE"],
    "MAE": -A_cv_score["test_MAE"]
})
A_cv_table.round(4)

In [ ]:
A_cv_table[["R2", "RMSE", "MAE"]].agg(["mean", "std"]).round(4)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(A_cv_table["Fold"], A_cv_table["R2"], marker="o")
plt.xlabel("Fold")
plt.ylabel("R²")
plt.title("Model A 5-fold CV")
plt.tight_layout()
plt.show()

In [ ]:
actual_vs_pred(
    y_test,
    A_test_pred,
    "Model A — OLS"
)

In [ ]:
A_residual = y_test.to_numpy() - A_test_pred

In [ ]:
residual_plot(
    A_test_pred,
    A_residual,
    "Model A residuals"
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(A_residual, bins=50)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Model A residual distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
stats.probplot(A_residual, dist="norm", plot=plt)
plt.title("Model A Q-Q plot")
plt.tight_layout()
plt.show()

In [ ]:
A_coef = pd.DataFrame({
    "Feature": A_KEEP,
    "Coefficient": A_model.coef_
})
A_coef["Absolute"] = A_coef["Coefficient"].abs()
A_coef = A_coef.sort_values("Absolute", ascending=False)
A_coef.head(15).round(4)

In [ ]:
tmp = A_coef.head(15).sort_values("Coefficient")
plt.figure(figsize=(8, 5))
plt.barh(tmp["Feature"], tmp["Coefficient"])
plt.xlabel("Standardized coefficient")
plt.title("Model A coefficients")
plt.tight_layout()
plt.show()

In [ ]:
A_coef.sort_values("Coefficient", ascending=False).head(8)[
    ["Feature", "Coefficient"]
].round(4)

In [ ]:
A_coef.sort_values("Coefficient").head(8)[
    ["Feature", "Coefficient"]
].round(4)

In [ ]:
A_condition_number = np.linalg.cond(XA_dev_scaled.to_numpy())
A_condition_number

In [ ]:
A_error = pd.DataFrame({
    "Actual": y_test,
    "Predicted": A_test_pred,
    "Residual": A_residual
})
A_error["Absolute error"] = A_error["Residual"].abs()

In [ ]:
A_error.sort_values(
    "Absolute error",
    ascending=False
).head(12).round(4)

In [ ]:
A_bins = pd.qcut(y_test, q=5, duplicates="drop")

In [ ]:
A_bin_result = (
    pd.DataFrame({
        "Actual": y_test,
        "Predicted": A_test_pred,
        "Bin": A_bins
    })
    .groupby("Bin", observed=True)
    .apply(lambda z: pd.Series({
        "N": len(z),
        "RMSE": np.sqrt(mean_squared_error(z["Actual"], z["Predicted"])),
        "MAE": mean_absolute_error(z["Actual"], z["Predicted"])
    }))
)
A_bin_result.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    range(len(A_bin_result)),
    A_bin_result["RMSE"],
    marker="o"
)
plt.xlabel("Target quintile")
plt.ylabel("RMSE")
plt.title("Model A error by target range")
plt.tight_layout()
plt.show()

In [ ]:
A_predictions = pd.DataFrame({
    "SAMPLE_ID": df_clean.iloc[test_idx]["SAMPLE_ID"].values,
    "Actual": y_test,
    "Predicted": A_test_pred,
    "Residual": A_residual
})

In [ ]:
A_predictions.to_csv(
    "LR_Model_A_predictions.csv",
    index=False
)

In [ ]:
joblib.dump(
    {
        "model": A_model,
        "imputer": A_imputer,
        "scaler": A_scaler,
        "features": A_KEEP
    },
    "LR_Model_A.joblib"
)

In [ ]:
A_final = pd.DataFrame([{
    "Model": "A",
    "Method": "OLS",
    "Raw features": len(A_FEATURES),
    **A_result
}])
A_final.round(4)

# MODEL B — Regularized Linear Regression
### Model A + PLIP

Model B adds interaction information and compares OLS, Ridge, Lasso and ElasticNet using development CV.

In [ ]:
XB = df_clean[B_FEATURES].apply(pd.to_numeric, errors="coerce")
XB = XB.replace([np.inf, -np.inf], np.nan)
XB.shape

In [ ]:
XB_dev = XB.iloc[dev_idx].reset_index(drop=True)
XB_test = XB.iloc[test_idx].reset_index(drop=True)

In [ ]:
pd.Series({
    "Raw features": len(B_FEATURES),
    "PLIP features": len(PLIP)
})

In [ ]:
XB_dev[PLIP].describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
B_plip_nonzero = (XB_dev[PLIP] > 0).mean().sort_values()
B_plip_nonzero.to_frame("Non-zero fraction")

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(B_plip_nonzero.index, B_plip_nonzero.values)
plt.xlabel("Non-zero fraction")
plt.title("Model B PLIP activity")
plt.tight_layout()
plt.show()

In [ ]:
def add_B_features(X):
    X = X.copy()
    X["FE.PLIPTotal"] = X[PLIP].sum(axis=1, min_count=1)
    X["FE.PLIPMean"] = X[PLIP].mean(axis=1)
    X["FE.PLIPStd"] = X[PLIP].std(axis=1)
    X["FE.PLIPActive"] = (X[PLIP] > 0).sum(axis=1)
    return X

In [ ]:
XB_dev_fe = add_B_features(XB_dev)
XB_test_fe = add_B_features(XB_test)
XB_dev_fe.shape

In [ ]:
B_missing = XB_dev_fe.isna().mean().sort_values(ascending=False)
B_missing.head(10).to_frame("Missing")

In [ ]:
B_constant = XB_dev_fe.nunique(dropna=True)
B_constant = B_constant[B_constant <= 1].index.tolist()
B_constant

In [ ]:
B_KEEP = [c for c in XB_dev_fe.columns if c not in B_constant]
len(B_KEEP)

In [ ]:
XB_dev_fe = XB_dev_fe[B_KEEP]
XB_test_fe = XB_test_fe[B_KEEP]

In [ ]:
B_diag_imputer = SimpleImputer(strategy="median")
XB_dev_imp = pd.DataFrame(
    B_diag_imputer.fit_transform(XB_dev_fe),
    columns=B_KEEP
)

In [ ]:
B_skew_before = XB_dev_imp.skew()
B_skew_before.abs().sort_values(ascending=False).head(12)

In [ ]:
B_power = PowerTransformer(
    method="yeo-johnson",
    standardize=False
)
XB_dev_power = pd.DataFrame(
    B_power.fit_transform(XB_dev_imp),
    columns=B_KEEP
)

In [ ]:
B_skew_after = XB_dev_power.skew()
pd.DataFrame({
    "Before": B_skew_before.abs(),
    "After": B_skew_after.abs()
}).sort_values("Before", ascending=False).head(12).round(3)

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(
    ["Before", "After"],
    [
        B_skew_before.abs().median(),
        B_skew_after.abs().median()
    ]
)
plt.ylabel("Median |skewness|")
plt.title("Model B Yeo-Johnson effect")
plt.tight_layout()
plt.show()

In [ ]:
B_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [ ]:
B_base_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("power", PowerTransformer(method="yeo-johnson")),
    ("scaler", RobustScaler())
]

In [ ]:
B_ols_pipe = Pipeline(
    B_base_steps + [
        ("model", LinearRegression())
    ]
)

In [ ]:
B_ols_cv = cross_validate(
    B_ols_pipe,
    XB_dev_fe,
    y_dev,
    cv=B_cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=N_JOBS
)

In [ ]:
B_ols_rmse = -B_ols_cv["test_score"].mean()
B_ols_rmse

In [ ]:
B_ridge_pipe = Pipeline(
    B_base_steps + [
        ("model", Ridge())
    ]
)

In [ ]:
B_ridge_grid = {
    "model__alpha": np.logspace(-4, 4, 30)
}

In [ ]:
B_ridge_search = GridSearchCV(
    B_ridge_pipe,
    B_ridge_grid,
    scoring="neg_root_mean_squared_error",
    cv=B_cv,
    n_jobs=N_JOBS
)

In [ ]:
B_ridge_search.fit(XB_dev_fe, y_dev)
B_ridge_search.best_params_

In [ ]:
-B_ridge_search.best_score_

In [ ]:
B_lasso_pipe = Pipeline(
    B_base_steps + [
        ("model", Lasso(max_iter=30000))
    ]
)

In [ ]:
B_lasso_grid = {
    "model__alpha": np.logspace(-5, 0, 24)
}

In [ ]:
B_lasso_search = GridSearchCV(
    B_lasso_pipe,
    B_lasso_grid,
    scoring="neg_root_mean_squared_error",
    cv=B_cv,
    n_jobs=N_JOBS
)

In [ ]:
B_lasso_search.fit(XB_dev_fe, y_dev)
B_lasso_search.best_params_

In [ ]:
-B_lasso_search.best_score_

In [ ]:
B_enet_pipe = Pipeline(
    B_base_steps + [
        ("model", ElasticNet(max_iter=30000))
    ]
)

In [ ]:
B_enet_grid = {
    "model__alpha": np.logspace(-4, 0, 14),
    "model__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9]
}

In [ ]:
B_enet_search = GridSearchCV(
    B_enet_pipe,
    B_enet_grid,
    scoring="neg_root_mean_squared_error",
    cv=B_cv,
    n_jobs=N_JOBS
)

In [ ]:
B_enet_search.fit(XB_dev_fe, y_dev)
B_enet_search.best_params_

In [ ]:
-B_enet_search.best_score_

In [ ]:
B_cv_compare = pd.DataFrame({
    "Method": ["OLS", "Ridge", "Lasso", "ElasticNet"],
    "CV_RMSE": [
        B_ols_rmse,
        -B_ridge_search.best_score_,
        -B_lasso_search.best_score_,
        -B_enet_search.best_score_
    ]
}).sort_values("CV_RMSE")
B_cv_compare.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(
    B_cv_compare["Method"],
    B_cv_compare["CV_RMSE"]
)
plt.ylabel("5-fold CV RMSE")
plt.title("Model B regularization comparison")
plt.tight_layout()
plt.show()

In [ ]:
B_best_name = B_cv_compare.iloc[0]["Method"]
B_best_name

In [ ]:
if B_best_name == "OLS":
    B_best_model = B_ols_pipe.fit(XB_dev_fe, y_dev)
elif B_best_name == "Ridge":
    B_best_model = B_ridge_search.best_estimator_
elif B_best_name == "Lasso":
    B_best_model = B_lasso_search.best_estimator_
else:
    B_best_model = B_enet_search.best_estimator_

In [ ]:
B_test_pred = B_best_model.predict(XB_test_fe)

In [ ]:
B_result = metrics(y_test, B_test_pred)
pd.Series(B_result).round(4)

In [ ]:
actual_vs_pred(
    y_test,
    B_test_pred,
    f"Model B — {B_best_name}"
)

In [ ]:
B_residual = y_test.to_numpy() - B_test_pred

In [ ]:
residual_plot(
    B_test_pred,
    B_residual,
    f"Model B — {B_best_name} residuals"
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(B_residual, bins=50)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Model B residual distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
stats.probplot(B_residual, dist="norm", plot=plt)
plt.title("Model B Q-Q plot")
plt.tight_layout()
plt.show()

In [ ]:
B_core = B_best_model.named_steps["model"]
B_coef = pd.DataFrame({
    "Feature": B_KEEP,
    "Coefficient": B_core.coef_
})
B_coef["Absolute"] = B_coef["Coefficient"].abs()
B_coef = B_coef.sort_values("Absolute", ascending=False)
B_coef.head(15).round(4)

In [ ]:
tmp = B_coef.head(15).sort_values("Coefficient")
plt.figure(figsize=(8, 5))
plt.barh(tmp["Feature"], tmp["Coefficient"])
plt.xlabel("Coefficient")
plt.title("Model B strongest coefficients")
plt.tight_layout()
plt.show()

In [ ]:
B_coef[
    B_coef["Feature"].str.startswith(("PLIP.", "FE.PLIP"))
].round(4)

In [ ]:
if hasattr(B_core, "coef_"):
    pd.Series({
        "Zero coefficients": int(np.sum(np.isclose(B_core.coef_, 0))),
        "Non-zero coefficients": int(np.sum(~np.isclose(B_core.coef_, 0)))
    })

In [ ]:
B_error = pd.DataFrame({
    "Actual": y_test,
    "Predicted": B_test_pred,
    "Residual": B_residual
})
B_error["Absolute error"] = B_error["Residual"].abs()
B_error.sort_values("Absolute error", ascending=False).head(12).round(4)

In [ ]:
AB_results = pd.DataFrame([
    {"Model": "A", "Method": "OLS", **A_result},
    {"Model": "B", "Method": B_best_name, **B_result}
])
AB_results.round(4)

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(AB_results["Model"], AB_results["R2"])
plt.ylabel("Locked test R²")
plt.title("Model A vs Model B")
plt.tight_layout()
plt.show()

In [ ]:
B_predictions = pd.DataFrame({
    "SAMPLE_ID": df_clean.iloc[test_idx]["SAMPLE_ID"].values,
    "Actual": y_test,
    "Predicted": B_test_pred,
    "Residual": B_residual
})
B_predictions.to_csv("LR_Model_B_predictions.csv", index=False)

In [ ]:
joblib.dump(
    {
        "model": B_best_model,
        "method": B_best_name,
        "features": B_KEEP
    },
    "LR_Model_B.joblib"
)

In [ ]:
B_final = pd.DataFrame([{
    "Model": "B",
    "Method": B_best_name,
    "Raw features": len(B_FEATURES),
    **B_result
}])
B_final.round(4)

# MODEL C — Full Linear Modelling
### Model B + VINA + P2

Model C uses the full feature representation. We compare full regularization, mutual-information feature selection and PCA.

In [ ]:
XC = df_clean[C_FEATURES].apply(pd.to_numeric, errors="coerce")
XC = XC.replace([np.inf, -np.inf], np.nan)
XC.shape

In [ ]:
XC_dev = XC.iloc[dev_idx].reset_index(drop=True)
XC_test = XC.iloc[test_idx].reset_index(drop=True)

In [ ]:
pd.Series({
    "Raw features": len(C_FEATURES),
    "VINA": len(VINA),
    "P2": len(P2)
})

In [ ]:
C_family_missing = pd.Series({
    "LIG": XC_dev[LIG].isna().mean().mean(),
    "MUT": XC_dev[MUT].isna().mean().mean(),
    "ENV": XC_dev[ENV].isna().mean().mean(),
    "PLIP": XC_dev[PLIP].isna().mean().mean(),
    "VINA": XC_dev[VINA].isna().mean().mean(),
    "P2": XC_dev[P2].isna().mean().mean()
})
C_family_missing.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(C_family_missing.index, C_family_missing.values)
plt.ylabel("Missing fraction")
plt.title("Model C missingness by family")
plt.tight_layout()
plt.show()

In [ ]:
def add_C_features(X):
    X = X.copy()

    X["FE.PLIPTotal"] = X[PLIP].sum(axis=1, min_count=1)

    X["FE.VinaMean"] = X[VINA].mean(axis=1)
    X["FE.VinaStd"] = X[VINA].std(axis=1)
    X["FE.VinaMin"] = X[VINA].min(axis=1)
    X["FE.VinaMax"] = X[VINA].max(axis=1)
    X["FE.VinaRange"] = X["FE.VinaMax"] - X["FE.VinaMin"]
    X["FE.VinaMedian"] = X[VINA].median(axis=1)

    p2 = [c for c in P2 if c.startswith("P2.")]
    p2dl = [c for c in P2 if c.startswith("P2dl.")]
    p2dp = [c for c in P2 if c.startswith("P2dp.")]

    if p2:
        X["FE.P2Total"] = X[p2].sum(axis=1, min_count=1)

    if p2dl:
        X["FE.P2dlTotal"] = X[p2dl].sum(axis=1, min_count=1)

    if p2dp:
        X["FE.P2dpTotal"] = X[p2dp].sum(axis=1, min_count=1)

    if p2dl and p2dp:
        X["FE.P2Contrast"] = X["FE.P2dlTotal"] - X["FE.P2dpTotal"]

    return X

In [ ]:
XC_dev_fe = add_C_features(XC_dev)
XC_test_fe = add_C_features(XC_test)
XC_dev_fe.shape

In [ ]:
C_missing = XC_dev_fe.isna().mean().sort_values(ascending=False)
C_missing.head(12).to_frame("Missing")

In [ ]:
C_diag_imputer = SimpleImputer(strategy="median")
XC_dev_imp = pd.DataFrame(
    C_diag_imputer.fit_transform(XC_dev_fe),
    columns=XC_dev_fe.columns
)

In [ ]:
C_var = XC_dev_imp.var()
C_constant = C_var[C_var <= 0].index.tolist()
C_constant

In [ ]:
C_KEEP = [c for c in XC_dev_fe.columns if c not in C_constant]
len(C_KEEP)

In [ ]:
XC_dev_fe = XC_dev_fe[C_KEEP]
XC_test_fe = XC_test_fe[C_KEEP]

In [ ]:
C_corr = XC_dev_imp[C_KEEP].corr().abs()
C_upper = C_corr.where(
    np.triu(np.ones(C_corr.shape), k=1).astype(bool)
)

In [ ]:
C_high_corr = C_upper.stack().sort_values(ascending=False)
C_high_corr.head(20)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(C_upper.stack().values, bins=50)
plt.xlabel("|Correlation|")
plt.ylabel("Feature pairs")
plt.title("Model C feature redundancy")
plt.tight_layout()
plt.show()

In [ ]:
C_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [ ]:
C_base_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]

In [ ]:
C_ridge_pipe = Pipeline(
    C_base_steps + [
        ("model", Ridge())
    ]
)

In [ ]:
C_ridge_grid = {
    "model__alpha": np.logspace(-5, 6, 40)
}

In [ ]:
C_ridge_search = GridSearchCV(
    C_ridge_pipe,
    C_ridge_grid,
    scoring="neg_root_mean_squared_error",
    cv=C_cv,
    n_jobs=N_JOBS
)

In [ ]:
C_ridge_search.fit(XC_dev_fe, y_dev)
C_ridge_search.best_params_

In [ ]:
-C_ridge_search.best_score_

In [ ]:
C_lasso_pipe = Pipeline(
    C_base_steps + [
        ("model", Lasso(max_iter=30000))
    ]
)

In [ ]:
C_lasso_grid = {
    "model__alpha": np.logspace(-5, 0, 20)
}

In [ ]:
C_lasso_search = GridSearchCV(
    C_lasso_pipe,
    C_lasso_grid,
    scoring="neg_root_mean_squared_error",
    cv=C_cv,
    n_jobs=N_JOBS
)

In [ ]:
C_lasso_search.fit(XC_dev_fe, y_dev)
C_lasso_search.best_params_

In [ ]:
-C_lasso_search.best_score_

In [ ]:
C_enet_pipe = Pipeline(
    C_base_steps + [
        ("model", ElasticNet(max_iter=30000))
    ]
)

In [ ]:
C_enet_grid = {
    "model__alpha": np.logspace(-4, 0, 12),
    "model__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9]
}

In [ ]:
C_enet_search = GridSearchCV(
    C_enet_pipe,
    C_enet_grid,
    scoring="neg_root_mean_squared_error",
    cv=C_cv,
    n_jobs=N_JOBS
)

In [ ]:
C_enet_search.fit(XC_dev_fe, y_dev)
C_enet_search.best_params_

In [ ]:
-C_enet_search.best_score_

In [ ]:
C_mi_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(mutual_info_regression)),
    ("model", Ridge())
])

In [ ]:
C_mi_grid = {
    "select__k": [30, 50, 70, 90, 110, "all"],
    "model__alpha": np.logspace(-3, 4, 16)
}

In [ ]:
C_mi_search = GridSearchCV(
    C_mi_pipe,
    C_mi_grid,
    scoring="neg_root_mean_squared_error",
    cv=C_cv,
    n_jobs=N_JOBS
)

In [ ]:
C_mi_search.fit(XC_dev_fe, y_dev)
C_mi_search.best_params_

In [ ]:
-C_mi_search.best_score_

In [ ]:
C_pca_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", Ridge())
])

In [ ]:
C_pca_grid = {
    "pca__n_components": [0.90, 0.95, 0.99],
    "model__alpha": np.logspace(-3, 4, 16)
}

In [ ]:
C_pca_search = GridSearchCV(
    C_pca_pipe,
    C_pca_grid,
    scoring="neg_root_mean_squared_error",
    cv=C_cv,
    n_jobs=N_JOBS
)

In [ ]:
C_pca_search.fit(XC_dev_fe, y_dev)
C_pca_search.best_params_

In [ ]:
-C_pca_search.best_score_

In [ ]:
C_cv_compare = pd.DataFrame({
    "Method": [
        "Ridge",
        "Lasso",
        "ElasticNet",
        "MI + Ridge",
        "PCA + Ridge"
    ],
    "CV_RMSE": [
        -C_ridge_search.best_score_,
        -C_lasso_search.best_score_,
        -C_enet_search.best_score_,
        -C_mi_search.best_score_,
        -C_pca_search.best_score_
    ]
}).sort_values("CV_RMSE")
C_cv_compare.round(4)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    C_cv_compare["Method"],
    C_cv_compare["CV_RMSE"]
)
plt.ylabel("5-fold CV RMSE")
plt.title("Model C method comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
C_best_name = C_cv_compare.iloc[0]["Method"]
C_best_name

In [ ]:
C_models = {
    "Ridge": C_ridge_search.best_estimator_,
    "Lasso": C_lasso_search.best_estimator_,
    "ElasticNet": C_enet_search.best_estimator_,
    "MI + Ridge": C_mi_search.best_estimator_,
    "PCA + Ridge": C_pca_search.best_estimator_
}

C_best_model = C_models[C_best_name]

In [ ]:
C_test_pred = C_best_model.predict(XC_test_fe)

In [ ]:
C_result = metrics(y_test, C_test_pred)
pd.Series(C_result).round(4)

In [ ]:
actual_vs_pred(
    y_test,
    C_test_pred,
    f"Model C — {C_best_name}"
)

In [ ]:
C_residual = y_test.to_numpy() - C_test_pred

In [ ]:
residual_plot(
    C_test_pred,
    C_residual,
    f"Model C — {C_best_name} residuals"
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(C_residual, bins=50)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Model C residual distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
stats.probplot(C_residual, dist="norm", plot=plt)
plt.title("Model C Q-Q plot")
plt.tight_layout()
plt.show()

In [ ]:
C_mi_best = C_mi_search.best_estimator_
C_selector = C_mi_best.named_steps["select"]
C_selected_features = np.array(C_KEEP)[C_selector.get_support()]
len(C_selected_features)

In [ ]:
C_selected_features[:20]

In [ ]:
C_family = pd.Series({
    "LIG": len([c for c in C_selected_features if c.startswith("LIG.")]),
    "MUT": len([c for c in C_selected_features if c.startswith("MUT.")]),
    "ENV": len([c for c in C_selected_features if c.startswith("ENV.")]),
    "PLIP": len([c for c in C_selected_features if c.startswith("PLIP.")]),
    "VINA": len([c for c in C_selected_features if c == "vina" or c.startswith("vina")]),
    "P2": len([c for c in C_selected_features if c.startswith(("P2.", "P2dl.", "P2dp."))]),
    "Engineered": len([c for c in C_selected_features if c.startswith("FE.")])
})
C_family

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(C_family.index, C_family.values)
plt.ylabel("Selected features")
plt.title("Model C selected feature families")
plt.tight_layout()
plt.show()

In [ ]:
C_error = pd.DataFrame({
    "Actual": y_test,
    "Predicted": C_test_pred,
    "Residual": C_residual
})
C_error["Absolute error"] = C_error["Residual"].abs()
C_error.sort_values("Absolute error", ascending=False).head(12).round(4)

In [ ]:
ABC_results = pd.DataFrame([
    {"Model": "A", "Method": "OLS", **A_result},
    {"Model": "B", "Method": B_best_name, **B_result},
    {"Model": "C", "Method": C_best_name, **C_result}
])
ABC_results.round(4)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(ABC_results["Model"], ABC_results["R2"])
plt.ylabel("Locked test R²")
plt.title("Linear Regression track — A vs B vs C")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(ABC_results["Model"], ABC_results["RMSE"])
plt.ylabel("Locked test RMSE")
plt.title("Linear Regression track — RMSE")
plt.tight_layout()
plt.show()

In [ ]:
ABC_change = pd.DataFrame({
    "Transition": ["A → B", "B → C", "A → C"],
    "Delta R2": [
        B_result["R2"] - A_result["R2"],
        C_result["R2"] - B_result["R2"],
        C_result["R2"] - A_result["R2"]
    ],
    "Delta RMSE": [
        B_result["RMSE"] - A_result["RMSE"],
        C_result["RMSE"] - B_result["RMSE"],
        C_result["RMSE"] - A_result["RMSE"]
    ]
})
ABC_change.round(4)

In [ ]:
C_predictions = pd.DataFrame({
    "SAMPLE_ID": df_clean.iloc[test_idx]["SAMPLE_ID"].values,
    "Actual": y_test,
    "Predicted": C_test_pred,
    "Residual": C_residual
})
C_predictions.to_csv("LR_Model_C_predictions.csv", index=False)

In [ ]:
ABC_results.to_csv(
    "LR_Model_A_B_C_results.csv",
    index=False
)

In [ ]:
joblib.dump(
    {
        "model": C_best_model,
        "method": C_best_name,
        "features": C_KEEP
    },
    "LR_Model_C.joblib"
)

In [ ]:
final_table = ABC_results.copy()
final_table["Raw features"] = [
    len(A_FEATURES),
    len(B_FEATURES),
    len(C_FEATURES)
]
final_table[
    [
        "Model",
        "Method",
        "Raw features",
        "R2",
        "RMSE",
        "MAE",
        "Pearson",
        "Spearman"
    ]
].round(4)

# Final interpretation

- **A** asks whether basic biochemical descriptors have a useful linear relationship with DDG.
- **B** adds PLIP interaction information and tests regularization.
- **C** adds the full VINA and P2 representation and tests stronger dimensionality control.

Select regularization and feature-processing choices using development CV. Use the locked test only for final evaluation.


## Export to the application
Run after all training cells. This verifies predictions against the locked test before exporting fitted preprocessing, models, feature lists, metrics, and plots.

In [ ]:
from backend.training.export import export_track
export_track(globals(), "linear_regression", PROJECT_ROOT)
